In [1]:
import gymnasium
import highway_env
import time
import os
import numpy as np
from Functions.Graphs import *
import gymnasium as gym
import numpy as np
from highway_env.vehicle.objects import Obstacle
from highway_env.vehicle.objects import Obstacle # Import a static object to anchor the camera

c:\Users\Claudio\Desktop\Python\ControlApplications\venv\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
sim_rate = 100
pol_rate = 12.5
sim_per = (sim_rate/pol_rate)
# 1. Instantiate the environment
env = gym.make(
    "racetrack-v0",
    render_mode="human",
    config={
        "screen_width": 900,
        "screen_height": 900,
        "action": {
            "type": "DiscreteMetaAction",
            # Increased target speeds to allow for a maximum speed of 20 m/s
            "target_speeds": [0, 5, 10, 15, 20, 25],
        },
        "other_vehicles": 0,
        "controlled_vehicles": 1,
        "duration": 100,
        "simulation_frequency": sim_rate,
        "policy_frequency": pol_rate,
        'show_trajectories': True,
        "centering_position": [0.15, 0.5],
        "scaling": 5,
    }
)

obs, info = env.reset()
camera_anchor = Obstacle(env.unwrapped.road, position=[0, 0])
env.unwrapped.viewer.observer_vehicle = camera_anchor

#obs, info = env.reset()
# Make the camera follow the main vehicle for better observation
#env.unwrapped.viewer.observer_vehicle = env.unwrapped.vehicle
# Use the Action Type's built-in helper to find indices
actions = env.unwrapped.action_type.actions_indexes
vehicle = env.unwrapped.vehicle
vehicle.speed = 0.0
vehicle.position[0] = 72
print(vehicle.position[0],vehicle.position[1])
laps = 0
previous_node = vehicle.lane_index[0]
phase = 'ACCELERATING'

print("Simulation started. Phase: ACCELERATING")

time_index = np.array([])
acc_ref = np.array([])
acc_cur = np.array([])
spd_ref = np.array([])
spd_cur = np.array([])
angle = np.array([])
k_gain = np.array([])
x_coords = np.array([])
y_coords = np.array([])
psi_history = []
beta_history = []

dt = 1.0 / pol_rate 
current_time = 0.0

MAX_SPEED = 20.0 # Define the desired maximum speed


while phase != 'FINISHED':
    

    kp_a = getattr(vehicle, 'KP_A', 0.6)
    current_acceleration = vehicle.action.get('acceleration', 0.0)
    current_steering_rad = vehicle.action.get('steering', 0.0)
    current_steering_deg = np.degrees(current_steering_rad)
    
    current_node = vehicle.lane_index[0]
    current_speed = vehicle.speed
    target_speed = vehicle.target_speed
    reference_acceleration = kp_a * (target_speed - current_speed)
    current_x = vehicle.position[0]
    current_y = vehicle.position[1]
    current_psi_rad = vehicle.heading
    current_psi_deg = np.degrees(current_psi_rad)
    current_beta_rad = np.arctan(0.5 * np.tan(current_steering_rad))
    current_beta_deg = np.degrees(current_beta_rad)

    psi_history.append(current_psi_deg)
    beta_history.append(current_beta_deg)
    x_coords = np.append(x_coords, current_x)
    y_coords = np.append(y_coords, current_y)
    acc_ref = np.append(acc_ref,reference_acceleration)
    acc_cur = np.append(acc_cur,current_acceleration)
    spd_ref = np.append(spd_ref,target_speed)
    spd_cur = np.append(spd_cur,current_speed)
    k_gain = np.append(k_gain,kp_a)
    angle = np.append(angle,current_steering_deg)
    time_index = np.append(time_index, current_time)



    print('Phase:',phase, 'Current Speed:',f'{current_speed:.2f}', 'Reference Acceleration:',f'{reference_acceleration:.2f}', 'Current Acceleration:',f'{current_acceleration:.2f}', 'Stirring Angle:',f'{current_steering_deg:.2f}')
    # --- LAP TRACKING LOGIC ---
    if previous_node == "i" and current_node == "a":
        laps += 1
        print(f"Lap boundary crossed! Total laps: {laps}")
    
    # --- PHASE & SPEED CONTROL LOGIC ---
    
    # Note: We no longer manually set vehicle.target_speed.
    # We let the environment's action handler manage it via FASTER/SLOWER.

    if phase == 'ACCELERATING':
        # If the vehicle's current target speed is less than MAX_SPEED, try to increase it.
        # The `DiscreteMetaAction` will handle incrementing `vehicle.target_speed`
        # through the `target_speeds` list.
        if vehicle.target_speed < MAX_SPEED:
            action = actions["FASTER"]
        else:
            # Once the target speed is MAX_SPEED, transition to cruising.
            # The vehicle will then try to reach and maintain this speed.
            phase = 'CRUISING'
            print(f"Target speed {MAX_SPEED} m/s set. Phase: CRUISING")
            action = actions["IDLE"] # Maintain the target speed

    elif phase == 'CRUISING':
        # Maintain speed around MAX_SPEED.
        # The `DiscreteMetaAction` will try to keep `vehicle.target_speed` at MAX_SPEED.
        # We use IDLE to tell the meta-action to not change the target speed.
        if laps >= 1: # Assuming 1 lap is enough to trigger braking
            phase = 'BRAKING'
            print('BRAKING', vehicle.position[0],vehicle.position[1])
            print("Approaching finish line. Phase: BRAKING")
            action = actions["SLOWER"] # Start braking by reducing target speed
        else:
            # If current speed deviates significantly from MAX_SPEED, adjust target speed.
            # This is a fine-tuning for the meta-action.
            if vehicle.speed < MAX_SPEED - 1.0:
                action = actions["FASTER"]
            elif vehicle.speed > MAX_SPEED + 1.0:
                action = actions["SLOWER"]
            else:
                action = actions["IDLE"] # Maintain current target speed

    elif phase == 'BRAKING':
        # If the vehicle's current target speed is greater than 0, try to decrease it.
        # The `DiscreteMetaAction` will handle decrementing `vehicle.target_speed`
        # through the `target_speeds` list.
        if vehicle.target_speed > 0:
            action = actions["SLOWER"]
        else:
            # Once the target speed is 0, wait for the vehicle to actually stop
            # and be at the finish line.
            action = actions["IDLE"] # Maintain target speed at 0
            if current_node == "a" and current_speed <= 0.1:
                phase = 'FINISHED'
                print('FINISHED', vehicle.position[0],vehicle.position[1])
                print(f"Vehicle stopped successfully. Final Speed: {current_speed:.2f} m/s")
            else:
                # If not at finish line 'a' yet, or not fully stopped, keep trying to slow down
                # (though target_speed is already 0, this ensures the action is still 'SLOWER'
                # if the vehicle hasn't reached 0 yet, or 'IDLE' if it has).
                action = actions["SLOWER"] if current_speed > 0.01 else actions["IDLE"]

    # Execute action
    obs, reward, done, truncated, info = env.step(action)
    env.render()
    
    previous_node = current_node
    current_time += dt

    if done or truncated:
        print("Vehicle crashed or execution timed out.")
        break

print("Simulation finished successfully.")
env.close()
#sim_duration = len(angle)/sim_per
#time_indx = np.linspace(0,sim_duration,len(angle))


72.0 5.0
Simulation started. Phase: ACCELERATING
Phase: ACCELERATING Current Speed: 0.00 Reference Acceleration: 16.67 Current Acceleration: 0.00 Stirring Angle: 0.00
Phase: ACCELERATING Current Speed: 0.63 Reference Acceleration: 7.28 Current Acceleration: 7.41 Stirring Angle: 0.00
Phase: ACCELERATING Current Speed: 1.18 Reference Acceleration: 6.37 Current Acceleration: 6.48 Stirring Angle: 0.00
Phase: ACCELERATING Current Speed: 1.66 Reference Acceleration: 5.57 Current Acceleration: 5.66 Stirring Angle: 0.00
Phase: ACCELERATING Current Speed: 2.08 Reference Acceleration: 4.87 Current Acceleration: 4.95 Stirring Angle: 0.00
Phase: ACCELERATING Current Speed: 2.45 Reference Acceleration: 4.25 Current Acceleration: 4.33 Stirring Angle: 0.00
Phase: ACCELERATING Current Speed: 2.77 Reference Acceleration: 3.72 Current Acceleration: 3.78 Stirring Angle: 0.00
Phase: ACCELERATING Current Speed: 3.68 Reference Acceleration: 10.54 Current Acceleration: 10.71 Stirring Angle: 0.00
Phase: ACCEL

In [3]:
df = pd.DataFrame()
df['time'] = time_index
df['x_coords'] = x_coords
df['y_coords'] = y_coords  
df['angle'] =  angle
df['psi'] =  psi_history
df['beta'] =  beta_history
df['speed'] =  spd_cur

out_dir = r'DyntheticDataset/'
os.makedirs(out_dir,exist_ok=True)
out_file = out_dir + 'RaceTrack11.csv'
df.to_csv(out_file,index=False)

In [207]:
# Extract road network
road_network = env.unwrapped.road.network

road_x = []
road_y = []

# Iterate through every connected segment in the network map
cont = 0
for node_from in road_network.graph:
    for node_to in road_network.graph[node_from]:
        for lane in road_network.graph[node_from][node_to]:
            # Sample coordinates along the lane at 1-meter intervals
            # lane.length gives the total length of this particular segment
            distances = np.linspace(0, lane.length, int(lane.length) + 1)
            
            for d in distances:
                # position(longitudinal, lateral) 
                # 0 lateral displacement represents the dead center of the lane
                coords = lane.position(d, 0)
                road_x.append(coords[0])
                road_y.append(coords[1])
                print(cont, coords)
                cont = cont+1

# Convert to numpy arrays for clean plotting


0 [42.  0.]
1 [43.  0.]
2 [44.  0.]
3 [45.  0.]
4 [46.  0.]
5 [47.  0.]
6 [48.  0.]
7 [49.  0.]
8 [50.  0.]
9 [51.  0.]
10 [52.  0.]
11 [53.  0.]
12 [54.  0.]
13 [55.  0.]
14 [56.  0.]
15 [57.  0.]
16 [58.  0.]
17 [59.  0.]
18 [60.  0.]
19 [61.  0.]
20 [62.  0.]
21 [63.  0.]
22 [64.  0.]
23 [65.  0.]
24 [66.  0.]
25 [67.  0.]
26 [68.  0.]
27 [69.  0.]
28 [70.  0.]
29 [71.  0.]
30 [72.  0.]
31 [73.  0.]
32 [74.  0.]
33 [75.  0.]
34 [76.  0.]
35 [77.  0.]
36 [78.  0.]
37 [79.  0.]
38 [80.  0.]
39 [81.  0.]
40 [82.  0.]
41 [83.  0.]
42 [84.  0.]
43 [85.  0.]
44 [86.  0.]
45 [87.  0.]
46 [88.  0.]
47 [89.  0.]
48 [90.  0.]
49 [91.  0.]
50 [92.  0.]
51 [93.  0.]
52 [94.  0.]
53 [95.  0.]
54 [96.  0.]
55 [97.  0.]
56 [98.  0.]
57 [99.  0.]
58 [100.   0.]
59 [42.  5.]
60 [43.  5.]
61 [44.  5.]
62 [45.  5.]
63 [46.  5.]
64 [47.  5.]
65 [48.  5.]
66 [49.  5.]
67 [50.  5.]
68 [51.  5.]
69 [52.  5.]
70 [53.  5.]
71 [54.  5.]
72 [55.  5.]
73 [56.  5.]
74 [57.  5.]
75 [58.  5.]
76 [59.  5.]
77 [60.

In [ ]:
import numpy as np

# 1. Criar um array com todos os índices possíveis (0 a 740)
todos_indices = np.arange(len(road_x))

# 2. Reunir todos os índices que você JÁ utilizou no seu plot
indices_plotados = np.concatenate([
    np.arange(0, 59),
    np.arange(119, 149),
    np.arange(190, 201),
    np.arange(212, 260),
    np.arange(326, 371),
    np.arange(410, 437),
    np.arange(466, 530),
    np.arange(609, 650),
    np.arange(722, 731)
])

# 3. Encontrar a diferença (o que sobrou)
indices_nao_plotados = np.setdiff1d(todos_indices, indices_plotados)

# 4. Extrair as coordenadas restantes
x_restante = [road_x[i] for i in indices_nao_plotados]
y_restante = [road_y[i] for i in indices_nao_plotados]

# Pronto! Agora x_restante e y_restante possuem as coordenadas que faltavam.
# Se quiser plotar o que ficou de fora usando sua função:
PlotPLY(x=x_restante, y=y_restante)

In [ ]:
x = road_x[59:118]+road_x[60:118]+road_x[150:189]+road_x[202:212]+road_x[261:324]+road_x[375:408]+road_x[437:466]+road_x[530:609]+road_x[660:718]+road_x[732:739] + [42]
y = road_y[59:118]+road_y[60:118]+road_y[150:189]+road_y[202:212]+road_y[261:324]+road_y[375:408]+road_y[437:466]+road_y[530:609]+road_y[660:718]+road_y[732:739] + [5]
x_outer = np.array(x)
y_outer  = np.array(y)


x = road_x[0:59]+road_x[119:149]+road_x[190:201]+road_x[212:260]+road_x[326:371]+road_x[410:437]+road_x[466:530]+road_x[609:650]+road_x[722:728] + [42]
y = road_y[0:59]+road_y[119:149]+road_y[190:201]+road_y[212:260]+road_y[326:371]+road_y[410:437]+road_y[466:530]+road_y[609:650]+road_y[722:728] + [0]

x_inner = np.array(x)
y_inner  = np.array(y)

x_mid,y_mid =

PlotSeriesPLY(xSeries=[x_outer,x_inner, x_mid],ySeries=[y_outer,y_inner,y_mid])



In [451]:
df = pd.DataFrame()
df['x'] = x
df['y'] = y  


out_dir = r'DyntheticDataset/'
os.makedirs(out_dir,exist_ok=True)
out_file = out_dir + 'InnerRace.csv'
df.to_csv(out_file,index=False)